# CaptionBERT sentence-address rotary screen
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/captionbert-trainer/blob/v0.2.0/notebooks/captionbert_rotary_screen.ipynb)

Select **G4 / RTX PRO 6000 (96 GB)**. This notebook installs tagged release **v0.2.0**
into an isolated environment compatible with Colab Python 3.13. It leaves Colab's
host packages intact. Each code cell is self-contained; run them in order.
The full screen uses four chunks, two epochs, batch 2048, A0–A3, seeds 1 and 2.
A2g is optional. Add the write-enabled Colab secret **HF_TOKEN** for Hub uploads.
Code stays on GitHub; weights and results go to AbstractPhil/captionbert-8192-rotary.


In [ ]:
def run_logged(command, log_path, env=None):
    import os
    from pathlib import Path
    import subprocess
    import time

    child_env = os.environ.copy() if env is None else dict(env)
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["PYTHONIOENCODING"] = "utf-8"
    path = Path(log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Streaming child output; console log: {path}", flush=True)
    started = time.monotonic()
    with path.open("a", encoding="utf-8", buffering=1) as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1,
                                   env=child_env)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            import signal
            process.send_signal(signal.SIGINT)
            raise
        finally:
            process.stdout.close()
        print(f"Child exited with code {returncode}; elapsed {time.monotonic() - started:.1f}s; "
              f"console log: {path}", flush=True)
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    return returncode

import os
import pathlib
import subprocess
import sys
import urllib.request

assert (3, 11) <= sys.version_info[:2] <= (3, 13), "Use a Python 3.11–3.13 runtime"
release = "v0.2.0"
env_dir = pathlib.Path("/content/captionbert-env")
if not (env_dir / "bin/python").exists():
    # Colab may omit stdlib ensurepip. virtualenv seeds pip without that module.
    bootstrap = pathlib.Path("/content/captionbert-bootstrap")
    run_logged([sys.executable, "-m", "pip", "install", "--target", str(bootstrap),
                    "virtualenv==20.38.0"], log_path="/content/captionbert-logs/setup.log")
    bootstrap_env = os.environ.copy()
    bootstrap_env["PYTHONPATH"] = str(bootstrap)
    run_logged([sys.executable, "-m", "virtualenv", "--no-download", str(env_dir)],
                   env=bootstrap_env, log_path="/content/captionbert-logs/setup.log")
python = str(env_dir / "bin/python")
# Download the three requirements files from the same immutable code tag.
req_dir = pathlib.Path("/content/captionbert-install")
req_dir.mkdir(parents=True, exist_ok=True)
for name in ("requirements.txt", "requirements-cuda.txt", "requirements-colab.txt"):
    url = f"https://raw.githubusercontent.com/AbstractEyes/captionbert-trainer/{release}/{name}"
    urllib.request.urlretrieve(url, req_dir / name)
run_logged([python, "-m", "pip", "install", "-r", str(req_dir / "requirements-colab.txt")], log_path="/content/captionbert-logs/setup.log")
run_logged([python, "-m", "pip", "install", "--no-deps",
                f"git+https://github.com/AbstractEyes/captionbert-trainer.git@{release}"], log_path="/content/captionbert-logs/setup.log")
run_logged([python, "-m", "pip", "check"], log_path="/content/captionbert-logs/setup.log")
run_logged([python, "-c", "import sys,torch,captionbert_rotary; print(sys.version); "
                "print(captionbert_rotary.__version__, torch.__version__, torch.version.cuda); "
                "assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"], log_path="/content/captionbert-logs/setup.log")


Run a bounded full-width engineering check before downloading data. This is synthetic
and does not measure sentence quality. Both seeds and all five supported arms run.


In [ ]:
def run_logged(command, log_path, env=None):
    import os
    from pathlib import Path
    import subprocess
    import time

    child_env = os.environ.copy() if env is None else dict(env)
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["PYTHONIOENCODING"] = "utf-8"
    path = Path(log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Streaming child output; console log: {path}", flush=True)
    started = time.monotonic()
    with path.open("a", encoding="utf-8", buffering=1) as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1,
                                   env=child_env)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            import signal
            process.send_signal(signal.SIGINT)
            raise
        finally:
            process.stdout.close()
        print(f"Child exited with code {returncode}; elapsed {time.monotonic() - started:.1f}s; "
              f"console log: {path}", flush=True)
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    return returncode

import subprocess
import textwrap

python = "/content/captionbert-env/bin/python"
run_logged([python, "-c", textwrap.dedent("""
    from captionbert_rotary.cli import smoke
    smoke(batch_size=8, steps=2, memory_limit_gb=20.0,
          output='/content/captionbert-smoke.json')
""")], log_path="/content/captionbert-logs/smoke.log")


Prepare only the five required chunks, about 4 GB of targets plus captions and token
caches. Tokenization precomputes sentence slots/anchors and reports rows reaching or
exceeding anchor 24. File hashes and exact row counts protect the caption/target join.


In [ ]:
def run_logged(command, log_path, env=None):
    import os
    from pathlib import Path
    import subprocess
    import time

    child_env = os.environ.copy() if env is None else dict(env)
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["PYTHONIOENCODING"] = "utf-8"
    path = Path(log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Streaming child output; console log: {path}", flush=True)
    started = time.monotonic()
    with path.open("a", encoding="utf-8", buffering=1) as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1,
                                   env=child_env)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            import signal
            process.send_signal(signal.SIGINT)
            raise
        finally:
            process.stdout.close()
        print(f"Child exited with code {returncode}; elapsed {time.monotonic() - started:.1f}s; "
              f"console log: {path}", flush=True)
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    return returncode

import subprocess
import textwrap

python = "/content/captionbert-env/bin/python"
run_logged([python, "-c", textwrap.dedent("""
    from captionbert_rotary.data import prepare
    prepare('/content/captionbert-data', chunks=(0, 1, 2, 3, 61), max_tokens=256)
""")], log_path="/content/captionbert-logs/prepare.log")


Train the screen. The literal v2 2,000-step warmup is retained, covering most of this
roughly 2,000-step screen. No gradient accumulation changes the 2,048 negative count.
A worst-case 256-token forward/backward/Adam probe runs before each arm. Completed
arms upload in the background with retries; resume restores already shipped runs.
Use the same configuration when resuming. A 4090 always keeps the 20 GB ceiling.

The cell streams merged stdout/stderr immediately and saves a console copy at
`/content/captionbert-logs/training.log`. The first step and every 25th step show
arm/seed, epoch/batch, step/total, loss/NCE/MSE, LR, gradient norm, throughput,
elapsed time, remaining training time, and CUDA memory. Validation, checkpoint,
benchmark, and upload stages print their own progress. Each step is also recorded
in `train.jsonl` in that arm/seed directory.

The current release also ships partial resume checkpoints after the first saved
training checkpoint and then every ten minutes. A lost runtime fetches its
latest compatible durable state; newer local progress wins. Upload outages
retain their queue and are listed in the final summary. Future arms collect
frozen rotary mechanism diagnostics and retain evaluation-step weights.


In [ ]:
def run_logged(command, log_path, env=None):
    import os
    from pathlib import Path
    import subprocess
    import time

    child_env = os.environ.copy() if env is None else dict(env)
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["PYTHONIOENCODING"] = "utf-8"
    path = Path(log_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Streaming child output; console log: {path}", flush=True)
    started = time.monotonic()
    with path.open("a", encoding="utf-8", buffering=1) as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1,
                                   env=child_env)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            import signal
            process.send_signal(signal.SIGINT)
            raise
        finally:
            process.stdout.close()
        print(f"Child exited with code {returncode}; elapsed {time.monotonic() - started:.1f}s; "
              f"console log: {path}", flush=True)
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    return returncode

import os
import subprocess
import textwrap
from google.colab import userdata

python = "/content/captionbert-env/bin/python"
child_env = os.environ.copy()
child_env["HF_TOKEN"] = userdata.get("HF_TOKEN")
if not child_env["HF_TOKEN"]:
    raise RuntimeError("Set the HF_TOKEN Colab secret with access to the result repository")
run_logged([python, "-c", textwrap.dedent("""
    import torch
    from captionbert_rotary import ScreenConfig
    from captionbert_rotary.train import run_screen
    assert torch.cuda.is_available()
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    limit = 80.0 if total_gb > 85 else 20.0
    cfg = ScreenConfig(
        arms=('A0', 'A1', 'A2', 'A3'),  # Add 'A2g' only for a separate declared screen.
        seeds=(1, 2), train_chunks=(0, 1, 2, 3), val_chunk=61,
        epochs=2, batch_size=2048, max_tokens=256,
        data_dir='/content/captionbert-data', output_dir='/content/captionbert-screen',
        memory_limit_gb=limit, push=True, benchmark=True,
    )
    run_screen(cfg)
""")], env=child_env, log_path="/content/captionbert-logs/training.log")


In [ ]:
import json
import pathlib

path = pathlib.Path("/content/captionbert-screen/summary.json")
print(json.dumps(json.loads(path.read_text()), indent=2))


## Monitor an already-running v0.1.0 session

Keep its training cell running. Open a separate Colab terminal and run:

```bash
curl -fLsS https://raw.githubusercontent.com/AbstractEyes/captionbert-trainer/v0.2.0/tools/watch_run.py -o /content/captionbert-watch.py
python -u /content/captionbert-watch.py /content/captionbert-screen
```

This reads existing files only, needs no package upgrade or GPU, and works while
the notebook kernel is busy. Ctrl+C in that terminal stops only the monitor.
An additional notebook cell will queue behind the busy training cell.
